# Setting Up TurtleBot 4

## Overview

This document explains the installation process needed to operate the TurtleBot 4 and have it ready for integration with the TC in our next chapter.

* Testbed Architecture (Chap. 0)
* Setting Up the TC (Chap. 1)
* Setting Up RPi Nodes (Chap. 2)
* Control Network Configuration (Chap. 3)
* Test Network Configuration (Chap. 4)

We also assume your TB4 and its Create 3 packages are already fully set up per [section/link]. By splitting the instructions into the steps below, this document aims to cover the network configuration needed for **remote multi-node movement**, which is the primary objective for the Ubiquitous Communications and Networking (UCaN) Lab.

---

## Requirements

| | Overhead Pi | TB4's Pi |
|---|---|---|
| Hardware | Raspberry Pi 4 Model B | Raspberry Pi 4 Model B |
| OS | Ubuntu Desktop 24.04.X LTS (64-bit) | Ubuntu Server 24.04 |
| ROS 2 | Jazzy | Jazzy |

---

In this setup, we have an overhead Pi with the default **username: ucanlab** and an IP on the typical lab subnet, **IP: 10.1.1.X** (as explained in Chap. 2). Inside the TB4, there is another Pi that has already been configured with Create 3 and ROS 2 Jazzy for control (as explained in [section/link]). To keep everything consistent with our lab work methodology, we keep the TB4 defaults: **username: ubuntu**, **IP: 192.168.185.3**. The monitor is connected via a micro-HDMI cable, and the keyboard and mouse are connected directly via USB-A cables to the USB 2.0 ports on the overhead Pi. The purpose is to double-check the connection with the TB4's Pi and to install the related software, which we discuss below.


This first section shows the instructions to install Robot Operating System 2 (ROS 2) Jazzy on the overhead Pi, because our TurtleBot 4's internal Pi mainly uses ROS 2 Jazzy on **Ubuntu Server 24.04**. The overhead Pi also needs to run the same ROS 2 version and operating system, since ROS 2 comes with rclpy, which is essential for us to communicate via the Data Distribution Service (DDS). Thus, the overhead Pi running on top of the TB4 should ideally be a **Raspberry Pi 4 Model B** with **Ubuntu Desktop 24.04.X LTS (64-bit)** specifically (i.e., any version of Ubuntu Desktop 24.04 works, such as 24.04.2, 24.04.3, 24.04.4, etc., since they all contain the `noble` base repositories that ROS 2 Jazzy is designed for). If you have already followed the steps in Chap. 2, the idea is basically the same: we flash the micro-SD card using Raspberry Pi Imager and select Ubuntu Desktop 24.04 as the OS version.

![Figure 2: Physical setup](./Images/setup_a.png)


After flashing the micro-SD card, boot up the overhead Pi with the monitor, keyboard, and mouse already connected. The next step is to connect the overhead Pi to our lab Internet router (i.e., ucanlabnet) and, via the Ubuntu terminal, install OpenSSH (since Ubuntu Desktop 24.04.4 doesn't come with it) and ROS 2.

- The commands below install openssh-server (i.e., for ssh, scp, etc.), which is essential for our remote connection purposes:
    - `sudo apt update`
    - `sudo apt install openssh-server`
    - `sudo systemctl enable --now ssh.socket ssh`
    - `sudo ss -tlnp | grep ':22'` $\qquad\quad$ [*optional: check if something is listening*]
    - `sudo ufw status`           $\qquad\qquad\qquad\qquad$ [*optional: check if ufw is active*]


- After installing openssh-server, our next step is to install ROS 2 Jazzy on the overhead Pi:
    - `sudo apt update`
    - `sudo apt install -y software-properties-common curl` $\qquad$ [*software tools for managing 3rd-party repos*]
    - `sudo add-apt-repository universe` $\qquad\qquad\qquad\qquad\qquad$ [*enable the universe repo*]
    - `export ROS_APT_SOURCE_VERSION=$(curl -s https://api.github.com/repos/ros-infrastructure/ros-apt-source/releases/latest | grep -F "tag_name" | awk -F\" '{print $4}')` [*add the ROS 2 apt source*]
    - `curl -L -o /tmp/ros2-apt-source.deb "https://github.com/ros-infrastructure/ros-apt-source/releases/download/${ROS_APT_SOURCE_VERSION}/ros2-apt-source_${ROS_APT_SOURCE_VERSION}.$(. /etc/os-release && echo $VERSION_CODENAME)_all.deb"`
    - `sudo dpkg -i /tmp/ros2-apt-source.deb`
    - `sudo apt update && sudo apt install -y ros-jazzy-ros-base` [*refresh and install*]
    - `source /opt/ros/jazzy/setup.bash`                          [*optional: activate the ROS 2 Jazzy environment*]
    - `python3 -c "import rclpy; print('rclpy ok')"`              [*optional: double-check that the package and environment are installed correctly; should print "rclpy ok"*]


After the installation, the overhead Pi and its robot must be on the same *ROS_DOMAIN_ID*; nodes on different domains cannot see each other. We can manually check or set the TB4's ROS_DOMAIN_ID by SSHing into the TB4's Pi with `ssh ubuntu@192.168.185.3` and typing `echo $ROS_DOMAIN_ID` (i.e., if the terminal prints 0 or nothing, set it using the next command) and `export ROS_DOMAIN_ID=1`. Then do the same for the overhead Pi, and verify that both have the same domain using `echo $ROS_DOMAIN_ID`. Alternatively, we can use the TB4's GUI by running the command `turtlebot4-setup` (the figure below describes the steps), then set the ROS_DOMAIN_ID to the same number on the overhead Pi.


![Figure 3: Physical setup](./Images/setup_b.png)


After all the packages are installed, we can check whether the overhead Pi is able to see the robot's ROS 2 topics by using the command `ros2 topic list | grep tb_01`. We should see the list of all topics, as shown in the image below.

![Figure 4: Physical setup](./Images/setup_c.png)


With that result, we conclude the TB4 setup.

## Next Chapter
[Integrate TB4 to the TC](https://github.com/UCaNLabUMB/Testbed_Controller/blob/main/Documentation/Setup_TC.md)